#### **CORD's ground_truth** field is a raw JSON string. You must parse it and construct a flat text target from the menu array

In [11]:
import json

def extract_cord_text(ground_truth_str: str) -> str:
    """
    Parse CORD ground_truth JSON and build a flat text string
    suitable as a TrOCR decoding target.
    
    CORD structure:
    {
      "gt_parse": {
        "menu": [
          {"nm": "Nasi Campur Bali", "cnt": "1 x", "price": "75,000"},
          ...
        ],
        "sub_total": {"subtotal_price": "428,000"},
        "total": {"total_price": "428,000"}
      }
    }
    """
    try:
        parsed = json.loads(ground_truth_str)
        gt = parsed.get("gt_parse", {})
        
        parts = []
        for item in gt.get("menu", []):
            nm    = item.get("nm", "").strip()
            cnt   = item.get("cnt", "").strip()
            price = item.get("price", "").strip()
            if nm:
                parts.append(f"{nm} {cnt} {price}".strip())
        
        # Optionally append total
        total = gt.get("total", {}).get("total_price", "")
        if total:
            parts.append(f"TOTAL {total}")
        
        return " | ".join(parts)
    
    except (json.JSONDecodeError, KeyError):
        return ""  # Skip malformed examples

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

#### **SROIE** has no single text field — you reconstruct it by joining the words list

In [ ]:
def extract_sroie_text(words: list) -> str:
    """
    SROIE provides a flat word list. Reconstruct as space-joined string.
    This is the OCR target: the full visible text of the receipt.
    """
    return " ".join(words)

### Combined Dataset Builder

In [ ]:
import json
from datasets import load_dataset, Dataset
from PIL import Image
import numpy as np

def build_combined_dataset():
    """
    Load CORD and SROIE, extract (image, text) pairs, combine into one Dataset.
    Returns a dict: {"train": Dataset, "validation": Dataset, "test": Dataset}
    """
    cord = load_dataset("naver-clova-ix/cord-v2")
    sroie = load_dataset("sizhkhy/SROIE")
    
    cord_splits = {}
    for split in ["train", "validation", "test"]:
        images, texts = [], []
        for ex in cord[split]:
            text = extract_cord_text(ex["ground_truth"])
            if text:  # Skip empty / malformed
                images.append(ex["image"])
                texts.append(text)
        cord_splits[split] = {"image": images, "text": texts}
    
    sroie_splits = {}
    for split in ["train", "test"]:
        images, texts = [], []
        for ex in sroie[split]:
            text = extract_sroie_text(ex["words"])
            if text:
                images.append(ex["images"])
                texts.append(text)
        sroie_splits[split] = {"image": images, "text": texts}
    
    # Combine: train = CORD train + SROIE train
    # validation = CORD validation only (SROIE has no val split)
    # test = CORD test + SROIE test
    combined = {
        "train": Dataset.from_dict({
            "image": cord_splits["train"]["image"] + sroie_splits["train"]["image"],
            "text":  cord_splits["train"]["text"]  + sroie_splits["train"]["text"],
        }),
        "validation": Dataset.from_dict({
            "image": cord_splits["validation"]["image"],
            "text":  cord_splits["validation"]["text"],
        }),
        "test": Dataset.from_dict({
            "image": cord_splits["test"]["image"] + sroie_splits["test"]["image"],
            "text":  cord_splits["test"]["text"]  + sroie_splits["test"]["text"],
        }),
    }
    
    print(f"Train size:      {len(combined['train'])}")
    print(f"Validation size: {len(combined['validation'])}")
    print(f"Test size:       {len(combined['test'])}")
    
    return combined

combined_dataset = build_combined_dataset()

Expected sizes (pre-augmentation):

Train: ~1,226 examples (626 CORD + 600 SROIE train)
Validation: ~100 examples (CORD val only)
Test: ~447 examples

#### **Augmentation** 
Apply augmentation only to training images, inline during the preprocess_trocr step. The augmentation simulates real-world receipt degradation.

In [ ]:
import albumentations as A
import cv2
import numpy as np
from PIL import Image

receipt_augmentation = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=(-0.3, 0.1), p=0.5),  # Thermal fade
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),                       # Scanner noise
    A.Rotate(limit=5, border_mode=cv2.BORDER_REPLICATE, p=0.5),        # Crumple tilt
    A.Perspective(scale=(0.02, 0.05), p=0.3),                          # Photo angle
    A.MotionBlur(blur_limit=3, p=0.2),                                 # Shaky photo
    A.ImageCompression(quality_lower=60, quality_upper=90, p=0.4),     # JPEG artifact
])

def apply_augmentation(pil_image: Image.Image) -> Image.Image:
    """Convert PIL → numpy → augment → PIL."""
    img_np = np.array(pil_image.convert("RGB"))
    augmented = receipt_augmentation(image=img_np)["image"]
    return Image.fromarray(augmented)

### Preprocessing Function

In [ ]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

def preprocess_trocr(example, augment: bool = False):
    """
    Preprocess a single (image, text) example for TrOCR training.
    
    Args:
        example: dict with keys 'image' (PIL Image) and 'text' (str)
        augment: apply receipt degradation augmentation (True for training only)
    
    Returns:
        dict with 'pixel_values' and 'labels' ready for Seq2SeqTrainer
    """
    image = example["image"]
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    image = image.convert("RGB")
    
    # Apply augmentation during training
    if augment:
        image = apply_augmentation(image)
    
    # Encode image → pixel_values (ViT expects 384x384)
    pixel_values = processor(images=image, return_tensors="pt").pixel_values
    
    # Encode text → token ids
    labels = processor.tokenizer(
        example["text"],
        padding="max_length",
        max_length=128,
        truncation=True,
    ).input_ids
    
    # Replace pad token id with -100 so it's ignored in cross-entropy loss
    labels = [
        token_id if token_id != processor.tokenizer.pad_token_id else -100
        for token_id in labels
    ]
    
    return {
        "pixel_values": pixel_values.squeeze(),
        "labels": labels,
    }

# Apply to datasets — augment train only
train_dataset = combined_dataset["train"].map(
    lambda ex: preprocess_trocr(ex, augment=True),
    remove_columns=["image", "text"],
    desc="Preprocessing train set",
)
val_dataset = combined_dataset["validation"].map(
    lambda ex: preprocess_trocr(ex, augment=False),
    remove_columns=["image", "text"],
    desc="Preprocessing val set",
)
test_dataset = combined_dataset["test"].map(
    lambda ex: preprocess_trocr(ex, augment=False),
    remove_columns=["image", "text"],
    desc="Preprocessing test set",
)

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

### Model Setup

In [ ]:
from transformers import VisionEncoderDecoderModel

model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")

# Required decoder config — without these the model won't generate properly
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

# Generation config — controls beam search during evaluation
model.config.eos_token_id  = processor.tokenizer.sep_token_id
model.config.max_new_tokens = 128
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

### Training Arguments

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr-smart-stock",
    
    # Training schedule
    num_train_epochs=10,
    per_device_train_batch_size=8,    # Safe for Kaggle T4 (16GB VRAM)
    per_device_eval_batch_size=8,
    
    # Optimizer
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    lr_scheduler_type="cosine",       # Better than linear for OCR tasks
    
    # Eval & saving
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,          # Lower CER = better
    save_total_limit=2,               # Keep only 2 checkpoints (Kaggle disk limit)
    
    # Generation
    predict_with_generate=True,
    generation_max_length=128,
    
    # Performance
    fp16=True,                        # Mixed precision — required on T4
    dataloader_num_workers=2,
    
    # Logging
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",                 # Disable wandb on Kaggle
)

### Metrics

In [ ]:
from jiwer import cer, wer

def compute_metrics(pred):
    pred_ids   = pred.predictions
    labels_ids = pred.label_ids
    
    # Decode predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    
    # Replace -100 (padding mask) with pad token id before decoding
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    
    return {
        "cer": cer(label_str, pred_str),
        "wer": wer(label_str, pred_str),
    }

### Trainer and Training

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor.tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

### Save & Export

In [ ]:
import os

# Save best model locally
trainer.save_model("./trocr-smart-stock/best-model")
processor.save_pretrained("./trocr-smart-stock/best-model")

# On Kaggle: save to /kaggle/working/ for persistence
save_path = "/kaggle/working/trocr-smart-stock-best"
trainer.save_model(save_path)
processor.save_pretrained(save_path)
print(f"Model saved to: {save_path}")

### Evaluate on Test Set

In [ ]:
results = trainer.evaluate(test_dataset)
print(f"Test CER: {results['eval_cer']:.4f}")
print(f"Test WER: {results['eval_wer']:.4f}")

# Target benchmarks:
# CER ≤ 0.05 (5%)
# WER ≤ 0.10 (10%)